# Screenshots of test patterns

**Copyright 2026 Martin Köhler**

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

KLayout screenshots of the test patterns of the appendix (KPEX/2.5D vs MAGIC), like the other test pattern figures:
black background, dotted grid, scale bar and the layer colors of the PDK, with the extracted areas marked in red (frame and hatch).

The figure includes (`figures/test_patterns/*.qmd`) write the capacitances onto them.

In [ ]:
import os
from typing import *
import xml.etree.ElementTree as ET

import klayout.db as kdb
import klayout.lay as klay

# NOTE: the klayout-pex checkout, next to this repository by default
KPEX_DIR = os.path.abspath(os.environ.get('KPEX_DIR', os.path.join('..', '..', 'klayout-pex')))
OUTPUT_DIR = os.path.abspath(os.path.join('..', 'figures', 'test_patterns'))
# NOTE: the display settings of the KLayout GUI the other test pattern figures were made with
KLAYOUTRC = os.path.abspath('test_pattern_screenshots.klayoutrc')


def load_technology(pdk: str):
    """
    The KLayout technology of KPEX for the PDK, whose layer properties give the colors
    (the layout view takes the only technology there is)
    """
    kdb.Technology.clear_technologies()
    tech = kdb.Technology.create_technology(pdk)
    tech.load(os.path.join(KPEX_DIR, 'pdk', pdk, 'libs.tech', 'kpex', f"{pdk}.lyt"))


def apply_klayoutrc(lv: klay.LayoutView, path: str):
    """
    Applies the settings of a klayoutrc (<config><key>value</key>…</config>) to the layout view
    """
    for setting in ET.parse(path).getroot():
        lv.set_config(setting.tag, (setting.text or '').strip())


def screenshot(pdk: str,
               pattern: str,
               aspect: str,
               view_box: kdb.DBox,
               marked_boxes: List[kdb.DBox],
               pixels_per_um: float = 17.6) -> str:
    """
    Saves the screenshot <pattern>_-_<aspect>.png of the test pattern, with the marked boxes (in µm)

    :return: the path of the screenshot
    """
    load_technology(pdk)
    lv = klay.LayoutView()
    lv.load_layout(os.path.join(KPEX_DIR, 'testdata', 'designs', pdk, 'test_patterns', f"{pattern}.gds.gz"))
    lv.max_hier()
    apply_klayoutrc(lv, KLAYOUTRC)
    lv.zoom_box(view_box)

    markers = []  # NOTE: the markers are shown as long as they exist
    for box in marked_boxes:
        m = klay.Marker(lv)
        m.set(box)
        m.color = 0xff0000
        m.frame_color = 0xff0000
        m.dither_pattern = 9  # lightly right-hatched
        m.line_width = 1
        markers.append(m)

    # NOTE: like the screenshots of the GUI on a HiDPI display: 2x pixels (fonts, lines),
    #       about as many pixels per µm as the other test pattern figures
    path = os.path.join(OUTPUT_DIR, f"{pattern}_-_{aspect}.png")
    lv.save_image_with_options(path,
                               width=round(view_box.width() * pixels_per_um),
                               height=round(view_box.height() * pixels_per_um),
                               resolution=0.5)
    return path

## Example: overlap_shield_same_net_*

Plates of 100 µm x 100 µm, `li1` and `met1` of net `A` under `met2` of net `B`, `met1` covering all or the left half of `li1`.
The marked areas are the overlaps of `met2` with `met1` and `li1`.
The view has 20 µm below the plates, for the scale bar and its text in the XXXLarge font, and 10 µm at the other sides.

In [ ]:
screenshot('sky130A', 'overlap_shield_same_net_100um_x_100um_li1_m1_m2', 'overlap',
           view_box=kdb.DBox(-10, -20, 110, 110),
           marked_boxes=[kdb.DBox(0, 0, 100, 100)])

In [ ]:
screenshot('sky130A', 'overlap_shield_same_net_half_100um_x_100um_li1_m1_m2', 'overlap',
           view_box=kdb.DBox(-10, -20, 110, 110),
           marked_boxes=[kdb.DBox(0, 0, 50, 100), kdb.DBox(50, 0, 100, 100)])